# Apache Spark Analytics
### Distributed Hacker News processing with Spark DataFrames

This notebook is optional. Install Java and `pyspark` before running. For small files, Pandas is often simpler; Spark is included for scalable processing and continuity with the original project.

In [ ]:
from pathlib import Path
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import *
spark = (SparkSession.builder
         .appName("HackerNewsAnalyticsPro")
         .config("spark.sql.session.timeZone", "UTC")
         .getOrCreate())
spark

## 1. Load the dataset
Use a real JSON Lines dataset if available; otherwise the bundled synthetic demo data.

In [ ]:
root = Path.cwd()
if not (root / "data").exists():
    root = root.parent
real_candidates = [root/"data"/"HNStories.jsonl", root/"data"/"HNStories.json"]
source = next((p for p in real_candidates if p.exists()), root/"data"/"sample"/"demo_stories.jsonl")
df = spark.read.json(str(source))
print("Source:", source)
print("Rows:", df.count())
df.printSchema()
df.show(5, truncate=False)

## 2. Normalize common field names

In [ ]:
cols = set(df.columns)
score_col = "score" if "score" in cols else ("points" if "points" in cols else None)
time_col = "time" if "time" in cols else ("created_at_i" if "created_at_i" in cols else None)
if score_col is None or time_col is None:
    raise ValueError("Dataset must include score/points and time/created_at_i fields.")
title_col = "title" if "title" in cols else None
url_col = "url" if "url" in cols else None
df = (df.withColumn("score_norm", F.col(score_col).cast("double"))
        .withColumn("epoch_seconds", F.col(time_col).cast("long")))
if title_col:
    df = df.withColumn("title_norm", F.coalesce(F.col(title_col).cast("string"), F.lit("")))
else:
    df = df.withColumn("title_norm", F.lit(""))
df = (df.withColumn("created_at_utc", F.to_timestamp(F.from_unixtime("epoch_seconds")))
        .withColumn("hour_utc", F.hour("created_at_utc"))
        .withColumn("weekday_utc", F.date_format("created_at_utc", "EEEE"))
        .withColumn("title_word_count", F.size(F.split(F.trim("title_norm"), r"\\s+")))
        .withColumn("is_high_performing", F.col("score_norm") > 200))
if url_col:
    df = df.withColumn("domain", F.lower(F.regexp_extract(F.col(url_col), r"^https?://(?:www\\.)?([^/]+)", 1)))
else:
    df = df.withColumn("domain", F.lit(""))
df = df.dropDuplicates(["id"]) if "id" in df.columns else df.dropDuplicates()
df.cache()
df.show(5, truncate=False)

## 3. Dataset profile and KPIs

In [ ]:
df.agg(
    F.count("*").alias("total_stories"),
    F.count("score_norm").alias("scored_stories"),
    F.avg("score_norm").alias("mean_score"),
    F.expr("percentile_approx(score_norm, 0.5)").alias("median_score"),
    F.countDistinct("domain").alias("unique_domains"),
    F.sum(F.col("is_high_performing").cast("int")).alias("high_performing_stories")
).show(truncate=False)

## 4. Story volume over time

In [ ]:
daily = (df.groupBy(F.to_date("created_at_utc").alias("date"))
           .agg(F.count("*").alias("story_count"))
           .orderBy("date"))
daily.show(30, truncate=False)

## 5. Hourly score and success-rate analysis

In [ ]:
hourly = (df.groupBy("hour_utc")
            .agg(F.count("*").alias("story_count"),
                 F.avg("score_norm").alias("mean_score"),
                 F.expr("percentile_approx(score_norm, 0.5)").alias("median_score"),
                 F.avg(F.col("is_high_performing").cast("double")).alias("high_performing_rate"))
            .orderBy("hour_utc"))
hourly.show(24, truncate=False)

## 6. Domain performance

In [ ]:
domains = (df.filter(F.col("domain") != "")
             .groupBy("domain")
             .agg(F.count("*").alias("story_count"),
                  F.avg("score_norm").alias("mean_score"),
                  F.avg(F.col("is_high_performing").cast("double")).alias("high_performing_rate"))
             .orderBy(F.desc("story_count")))
domains.show(25, truncate=False)

## 7. Title length

In [ ]:
title_metrics = (df.groupBy("title_word_count")
                   .agg(F.count("*").alias("story_count"),
                        F.avg("score_norm").alias("mean_score"),
                        F.avg(F.col("is_high_performing").cast("double")).alias("high_performing_rate"))
                   .orderBy("title_word_count"))
title_metrics.show(50, truncate=False)

## 8. Export aggregated results

In [ ]:
output = root / "reports" / "spark"
output.mkdir(parents=True, exist_ok=True)
hourly.coalesce(1).write.mode("overwrite").option("header", True).csv(str(output/"hourly"))
domains.limit(100).coalesce(1).write.mode("overwrite").option("header", True).csv(str(output/"domains"))
print("Spark aggregates exported to", output)

## 9. Resource cleanup
Stop the Spark session when finished.

In [ ]:
df.unpersist()
spark.stop()